# DOLFINx/UFL with independent sparse solvers

DOLFINx supplies native finite-element assembly; `SolverConfig` selects the local and global linear solvers independently. This notebook prefers PARDISO when its optional MKL runtime is installed, and executes SciPy configurations explicitly for comparison. None of these configurations requests PETSc.

The analytical Darcy case has identity permeability on the unit square, two macrotriangles, independently refined continuous P1 pressure spaces, and one constant trace coefficient per oriented macroface. The exact pressure, physical Darcy flux and source are

$$
\begin{aligned}
p(x,y)&=1+x+2y,\\
q&=-\nabla p=(-1,-2),\\
f&=\nabla\cdot q=0.
\end{aligned}
$$

Dirichlet pressure data equal this affine field on the entire outer boundary. The imported `DarcyProvider` writes the signed UFL volume and boundary forms on each worker's `MPI.COMM_SELF` mesh. A physical integral selects the local complement of the constant kernel, while the constant pressure mode remains a global unknown:

$$
\begin{aligned}
(\nabla p_T,\nabla v)_T
 +\langle\lambda_T,v\rangle_{\partial T}&=0,\\
\lambda_F&=q\cdot n_F,\\
\left.\lambda_T\right|_F&=(n_T\cdot n_F)\lambda_F,\\
\int_T p_T^{\mathrm{lift}}\,dx&=0.
\end{aligned}
$$

Here $n_F$ is the fixed global face normal and $n_T$ is the outward normal of macrocell $T$. The global multiplier $\lambda_F$ is the physical oriented Darcy normal flux; its signed local restriction $\lambda_T$ is the outward normal flux. Both local trace pairings use these declared signs; no Robin or pseudo-traction multiplier is introduced.

## Reproducible environment

From the checkout, use the checked-in Pixi lockfile:

```bash
pixi run --locked -e introduction-intel notebooks-run foundations/operators/dolfinx_sparse_solvers.ipynb
```

For interactive use, start Jupyter in `introduction-intel` and select its Python kernel. This environment contains DOLFINx/UFL, MPI, SciPy, MKL/PARDISO and the notebook tools. DOLFINx JIT assembly needs the platform's C compiler; Windows uses the Visual Studio compiler environment described in `docs/windows.md`.

Without `pypardiso`, the availability report is explicit and only the SciPy configuration runs. If an installed native backend cannot initialize or satisfy its numerical residual criterion, the error propagates. The notebook makes no timing or speedup claim.

A DOLFINx distribution built with PETSc support may initialize `petsc4py` during its own import when that package is installed. The provenance below records those native build capabilities; the selected linear adapters remain SciPy and PARDISO.

In [ ]:
from pathlib import Path
import importlib.util
import os
import platform
import sys

import numpy as np
from IPython.display import display

ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file()
    and (path / "src/pymhm").is_dir()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from examples.variational_darcy import DarcyProvider, affine_pressure, build_problem
from pymhm import ExecutionConfig, SolverConfig, assemble
from pymhm.core.multiscale import MultiscaleSolution, MultiscaleSystem


In [ ]:
import dolfinx
import scipy
import ufl

pardiso_available = importlib.util.find_spec("pypardiso") is not None
if pardiso_available:
    import pypardiso
    pardiso_version = pypardiso.__version__
else:
    pardiso_version = "not installed; PARDISO configurations will not execute"

provenance = {
    "platform": platform.platform(),
    "python": platform.python_version(),
    "dolfinx": dolfinx.__version__,
    "ufl": ufl.__version__,
    "numpy": np.__version__,
    "scipy": scipy.__version__,
    "pypardiso": pardiso_version,
    "native_threads_per_local_worker": 1,
    "assembly": "native CSR and vectors, real binary64, COMM_SELF",
    "dolfinx_build_has_petsc": dolfinx.has_petsc,
    "dolfinx_build_has_petsc4py": dolfinx.has_petsc4py,
}
display(provenance)


## Separate assembly, local elimination and global solution

`build_problem(provider="fenics")` returns the reusable, importable Darcy provider and its global `Equation`. It preserves the operator, mesh, kernel, physical moments, boundary data and trace orientation for every solver configuration.

The first configuration uses PARDISO for both stages when available. The other configurations demonstrate that local condensation and the global skeleton solve can use different adapters. SciPy uses its direct sparse solver. All adapters retain the package's existing residual tolerances and resource cleanup.

In [ ]:
problem = build_problem(provider="fenics", subdivisions=2)
assert isinstance(problem.local_provider, DarcyProvider)
macro = problem.local_provider.macro_mesh

exactq = np.array([-1.0, -2.0])
expected_trace = macro.normals @ exactq
expected_means = affine_pressure(macro.points[macro.cells].mean(axis=1))

configurations = []
if pardiso_available:
    configurations.extend([
        SolverConfig(local_solver="pypardiso", global_solver="pypardiso"),
        SolverConfig(local_solver="pypardiso", global_solver="scipy"),
        SolverConfig(local_solver="scipy", global_solver="pypardiso"),
    ])
configurations.append(SolverConfig(local_solver="scipy", global_solver="scipy"))
preferred = configurations[0]
print("Preferred local/global solvers:", preferred.local_solver, preferred.global_solver)


In [ ]:
def verify_physical_fields(
    system: MultiscaleSystem,
    solution: MultiscaleSolution,
) -> dict[str, object]:
    """Check affine pressure, oriented physical flux and integral pressure means."""
    pressure_error = max(
        float(np.max(np.abs(field - affine_pressure(metadata["points"]))))
        for field, metadata in zip(solution.fields, system.local_metadata, strict=True)
    )
    flux_error = float(np.max(np.abs(solution.trace - expected_trace)))
    means = np.array([
        float(response.problem.constraints[:, 0] @ field)
        / float(response.problem.constraints[:, 0].sum())
        for response, field in zip(system.responses, solution.fields, strict=True)
    ])
    mean_error = float(np.max(np.abs(means - expected_means)))
    assert pressure_error < 1e-11
    assert flux_error < 1e-11
    assert mean_error < 1e-11
    assert solution.raw_residual is not None and solution.raw_residual < 1e-11
    return {
        "local_solver": system.solvers.local_solver,
        "global_solver": system.solvers.global_solver,
        "max_pressure_coefficient_error": pressure_error,
        "max_oriented_Darcy_flux_error": flux_error,
        "max_physical_mean_error": mean_error,
        "pressure_means": means.tolist(),
        "raw_original_equation_relative_residual": solution.raw_residual,
    }

serial_execution = ExecutionConfig("serial", native_threads=1)
serial_solutions = {}
serial_systems = {}
records = []
for configuration in configurations:
    system = assemble(problem, execution=serial_execution, solvers=configuration)
    solution = system.solve()
    key = (configuration.local_solver, configuration.global_solver)
    serial_systems[key] = system
    serial_solutions[key] = solution
    records.append(verify_physical_fields(system, solution))

display(records)


In [ ]:
scipy_reference = serial_solutions[("scipy", "scipy")]
for solution in serial_solutions.values():
    np.testing.assert_allclose(solution.trace, scipy_reference.trace, atol=1e-11, rtol=0)
    for field, reference in zip(solution.fields, scipy_reference.fields, strict=True):
        np.testing.assert_allclose(field, reference, atol=1e-11, rtol=0)

preferred_key = (preferred.local_solver, preferred.global_solver)
preferred_solution = serial_solutions[preferred_key]

print("face    normal_x    normal_y    exact q.n    solved lambda")
for face, (normal, expected, computed) in enumerate(
    zip(macro.normals, expected_trace, preferred_solution.trace, strict=True)
):
    print(f"{face:4d} {normal[0]:11.6f} {normal[1]:11.6f} {expected:12.6f} {computed:16.6f}")
print("Exact pressure means:", expected_means)
print("Computed pressure means:", records[0]["pressure_means"])
print("All executed configurations agree in the same native pressure coefficient order.")


## Spawn workers own their native meshes and solver state

The same importable provider can assemble and condense inside spawned processes. It contains portable mesh and skeleton data; DOLFINx/UFL handles are created inside each call and only assembled coefficients and numerical reconstruction metadata return to the coordinator. The global solve still uses the independently selected global adapter.

The run below submits the two macrotriangles as separate batches to a pool with two workers and one native numerical thread per local worker. Process identifiers verify that native assembly occurred outside the notebook process. This is a portability and numerical agreement check.

In [ ]:
spawn_execution = ExecutionConfig(
    "process", workers=2, native_threads=1, batch_size=1,
)
spawn_system = assemble(problem, execution=spawn_execution, solvers=preferred)
spawn_solution = spawn_system.solve()
spawn_record = verify_physical_fields(spawn_system, spawn_solution)

assembly_pids = sorted({record["assembly_pid"] for record in spawn_system.local_metadata})
assert assembly_pids and os.getpid() not in assembly_pids
np.testing.assert_allclose(spawn_solution.trace, preferred_solution.trace, atol=1e-11, rtol=0)
for field, reference in zip(spawn_solution.fields, preferred_solution.fields, strict=True):
    np.testing.assert_allclose(field, reference, atol=1e-11, rtol=0)

spawn_record.update({
    "execution": "spawn",
    "configured_workers": 2,
    "native_assembly_outside_coordinator": os.getpid() not in assembly_pids,
    "processes_that_assembled_macrocells": len(assembly_pids),
})
display(spawn_record)

assert preferred.local_solver in {"scipy", "pypardiso"}
assert preferred.global_solver in {"scipy", "pypardiso"}
print("Executed local/global adapters:", preferred.local_solver, preferred.global_solver)


## Scientific scope

The affine pressure lies exactly in the executed P1 local spaces. The gradient and the physical Darcy flux are constant, and volume, face and pressure-moment integrands have polynomial degrees covered by their assembly quadrature. The source follows by independently differentiating the exact flux. The integral pressure checks verify the physical moments, and the signed face table verifies the normal convention.

This analytical patch verifies the native assembly/solver composition and reconstruction in the stated spaces. Its original-equation residual is an algebraic diagnostic; the notebook does not establish convergence rates, general inf-sup stability or a matched literature reproduction. Primal pressure reconstruction does not supply a general H(div) flux: this affine case happens to have a constant physical flux, and it cannot support a claim of fine-cell conservation for arbitrary primal solutions.

The package source notebook has no stored outputs. Executed copies and their provenance are written under `build/notebooks/foundations/operators/`.